<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-01-assistant-configuration/notebook.ipynb)


# Session 1 — Configure the assistant and the repository instructions

**Goal:** turn a generic assistant into a project-aware collaborator, and practice the inspect → plan → edit → test → review loop.

This session runs in your coding assistant, not in this notebook. The notebook is your checklist and logbook.

In [8]:
# manual-run: edits assistant configuration — run with your assistant open
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = ollama (qwen2.5:7b-instruct at http://localhost:11434/v1)
ready. LIVE is the ollama lane.


In [9]:
from bootcamp_agent.checks import check, review

## An assistant that lives in this notebook

Session 1 is about the assistant that edits your files, and that one runs
outside the notebook. This is a different and much smaller thing: a coach that
answers **from the course's own pages**, in this kernel, with no key, no
network, and no second window.

It quotes the page it used. A page id is something you can open; a confident
paraphrase is not.

It only knows the weeks that are in **your** clone, so if you ask about a week
that has not opened yet it will say so rather than invent it.

In [10]:
from bootcamp_agent.coach import coach

coach("which sessions need a coding assistant?")

# Try your own. These land well today:
#   coach("how do I install uv")
#   coach("do I need an API key")
#   coach("can I use colab")
#
# And one that should REFUSE, because nothing in the pages supports it:
#   coach("what is the capital of Peru")
#
# It is NOT always right, and it got worse this week. Ask it "what is AGENTS.md
# for?" and the top hit is the bonus page about improving the coach — a page
# added two days ago, which now outranks the pages that actually answer. Adding
# one document degraded retrieval for questions it has nothing to do with.
# That is measurable, and beating it is an open pull request — see the bonus unit.

--- Give the course to your assistant  [unit0/ask-your-assistant]
- It is already there — `from bootcamp_agent.coach import coach`
- Ask — `coach("what is AGENTS.md for?")`
- It prints the passages **and the page id each came from**. A page id is something you can open; a paraphrase is not
- It only knows the weeks in **your** clone. Ask about an unpublished week and it says so instead of inventing one
- No key, no network, no model. It quotes pages, it does not write prose

## In a terminal

The coach is its own small program, and `uvx` runs it straight from GitHub —
nothing to install, nothing to clone, no PyPI.

```bash
uvx --from "git+https://github.com/Gecko-Academy/gecko-ai-coach" \
  ai-coach ask "which sessions need a coding assistant" --pages ./units/en
```

--- Instructions are code  [unit1/session-01-assistant-configuration/concepts-2]
Read this repository's `AGENTS.md` against that table. Every row is present.
The "Do not" rows sit under **Coding rules** and **Safety**: no 

## 1. Warm-up: weak prompt vs project-aware prompt

The same task, asked two ways. **Run this here** — no assistant, no key, no
second window. It uses whichever lane your `.env` names.

- **Weak:** *"add a search feature"*
- **Project-aware:** *"read AGENTS.md, then propose a plan to add a tags filter to `search_documents` in `src/bootcamp_agent/tools.py` — plan only, no edits"*

**What to look for.** The weak prompt has to invent a codebase: it will name
files that do not exist here and pick an architecture nobody asked for. The
project-aware one is bounded — one file, one change, and permission to do
nothing else.

**On the `fake` lane both answers are identical**, because `FakeLLM` returns the
same canned string whatever you ask. That is a correct run and a boring lesson.
For the real contrast, point `.env` at a local model — `make ollama`, then
`BOOTCAMP_PROVIDER=ollama`. See [a local model](../../unit0/local-model.mdx).
It is free and it stays on your machine.

In [11]:
# ---------------------------------------------------------------------
# THIS RUNS AS SHIPPED. On `fake` both answers come back identical — that is
# FakeLLM being deterministic, not the prompts being equivalent.
# To stand above it: run it again on a real lane and paste what changed.
# ---------------------------------------------------------------------
from bootcamp_agent.config import load_settings
from bootcamp_agent.llm import get_client

WEAK = "add a search feature"
PROJECT_AWARE = (
    "read AGENTS.md, then propose a plan to add a tags filter to "
    "search_documents in src/bootcamp_agent/tools.py — plan only, no edits"
)

settings = load_settings()
client = get_client(settings)
print(f"lane: {settings.provider}\n")

for label, prompt in (("WEAK", WEAK), ("PROJECT-AWARE", PROJECT_AWARE)):
    print(f"--- {label}")
    print(client.complete(system="You are a careful engineer.", user=prompt)[:600])
    print()

lane: ollama

--- WEAK
Sure, I can help you add a search feature to a system or application. To give you the best advice, could you please specify a bit more about the context? Here are some questions to consider:

1. **What type of system or application is it?** (e.g., web application, mobile app, desktop application)
2. **What kind of data do you want to search?** (e.g., text, images, products, user profiles)
3. **Are there any existing search functionalities that can be improved or extended?**
4. **What is the expected user experience?** (e.g., real-time search, autocomplete, filters)

Once you provide more detai

--- PROJECT-AWARE
To add a tags filter to the `search_documents` function in `src/bootcamp_agent/tools.py`, follow this plan:

1. **Review AGENTS.md**: Understand the structure and functionality of the `search_documents` function and any existing filters.
   
2. **Identify Entry Points**:
   - Determine where the `search_documents` function is called and how documents are 

## 2. The task loop, enforced by you

Feature: **add an optional `tags` filter to `search_documents`** — work in a scratch branch/copy.

- [ ] Ask for a **plan** first. Read it. Restrict files it may touch.
- [ ] Ask for the **smallest implementation**.
- [ ] Inspect the **diff** yourself, line by line.
- [ ] Verify with what this repository gives you: `uv run ruff check src/bootcamp_agent/tools.py`,
      then the `check(...)` cell below. (There is no `pytest` here — the suite holds the solved
      value of every exercise and is never published. `AGENTS.md` says so too.)
- [ ] **Reject at least one change** — unsafe, unnecessary, or out of scope — and record what you rejected and why below.
- [ ] Ask for a summary of remaining risks.

**Rejected change + reason:** *I rejected the behavior when tags is an empty array, instead of throwing an error, it should not apply any filtering by tags. The reason is because I think it should be understanding as I don't want to filter the documents by any tag, therefore give me all of them*

## 3. Improve the instructions

Where did the assistant assume wrong? That sentence belongs in `AGENTS.md`. Make the edit, note it here — instructions are code (see `docs/guides/harness-engineering.md`).

In [17]:
"Confirm the behaviors expected for each possible values that each input tool could have."

'Confirm the behaviors expected for each possible values that each input tool could have.'

## 4. The loop, applied to this course

From tomorrow on, every exercise ends with a `check(...)` cell. Give your assistant the exercise's **Context** and **Instructions**, let it fill the `TODO(you)` lines, then run the check yourself. You read the verdict, not the assistant.

## 5. Exercise: the task loop, evidenced

**Context.** The loop only works if you can show you ran it. Four pieces of evidence, and the third one is the whole session: a change you refused. If you rejected nothing, you were not reviewing.

**Instructions.**

1. Work the feature (a `tags` filter on `search_documents`) in a scratch copy, through plan, edit, test, review.
2. Fill each field from what actually happened. `plan_approved` is filled as an example; replace it with yours.
3. Run the check. It refuses a blank rejection, because a loop with no rejection is not the loop.

In [18]:
# ---------------------------------------------------------------------
# THIS RUNS AS SHIPPED, and passes its check. That is the floor.
# But of every cell in this course, THIS is the one where shipped text is
# worth the least: it is a record of somebody else's session, not yours.
# To stand above it: replace all five with what YOUR assistant actually did —
# especially `rejected_change`, which is the whole session.
# ---------------------------------------------------------------------
loop = {
    "plan_approved": "It proposed to edit just the tools.py: add an optional tags argument as a list or None by default, filter the documents when tags exists.",
    "diff_inspected": "Twelve lines in search_documents . I read the filter placement: it runs after retrieve, so the cap still applies.",
    "rejected_change": "Assumed empty tags throws an error.",
    "why_rejected": "Behavior didn't confirm on how to treat or validate inputs",
    "risks": "It leaverages an expected behavior",
}
for key, value in loop.items():
    print(f"{key:18} {'(empty)' if not value else value[:58]}")

plan_approved      It proposed to edit just the tools.py: add an optional tag
diff_inspected     Twelve lines in search_documents . I read the filter place
rejected_change    Assumed empty tags throws an error.
why_rejected       Behavior didn't confirm on how to treat or validate inputs
risks              It leaverages an expected behavior


**Expected output** (yours may differ in wording, not in shape):

```
plan_approved      It proposed editing only tools.py: add an optional tags
diff_inspected     Six lines in search_documents plus one new test. I read
...
✅ ch01-e1 passed
```

In [19]:
check("ch01-e1", loop)

✅ ch01-e1 passed


True

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [20]:
review("ch01")

ch01: 1/1 passed  ·  100/100 marks


True

## Exit ticket

Homework: keep the improved instruction file; bring the rejected-change story to tomorrow's warm-up.